In [0]:
from pyspark.sql.functions import *

customers = spark.table("silver.silver_customers")
orders = spark.table("silver.silver_orders")
order_items = spark.table("silver.silver_order_items")
products = spark.table("silver.silver_products")
payments = spark.table("silver.silver_payments")

In [0]:
order_details = orders \
    .join(order_items, "order_id") \
    .join(products, "product_id")

order_details.show(5)

In [0]:
order_details = order_details.withColumn(
    "total_price",
    col("quantity") * col("price")
)

revenue_df = order_details.groupBy("order_id") \
    .agg(
        sum("total_price").alias("order_revenue")
    )

In [0]:
top_products = order_details.groupBy("product_name") \
    .agg(
        sum("quantity").alias("total_sold"),
        sum("total_price").alias("revenue")
    ) \
    .orderBy(desc("revenue"))

top_products.show(10)

In [0]:
customer_revenue = order_details \
    .join(customers, "customer_id") \
    .groupBy("customer_name") \
    .agg(
        sum("total_price").alias("total_spent")
    ) \
    .orderBy(desc("total_spent"))

customer_revenue.show(10)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

revenue_df.write.mode("overwrite").saveAsTable("gold.gold_revenue")

top_products.write.mode("overwrite").saveAsTable("gold.gold_top_products")

customer_revenue.write.mode("overwrite").saveAsTable("gold.gold_customer_revenue")